### Exam 2

# EXAM PROBLEM - Vehicle Purchase Optimization

A city agency is planning a new fleet purchase. It is considering four vehicle types:

- Small Compact
- Medium Sedan
- Large Pickup Truck
- SUV

The agency wants to maximize total annual service value, subject to a budget constraint and an average MPG requirement.

Use Python in your Jupyter notebook with OR-Tools to build and solve this linear optimization model.

---

## Vehicle Data

| Vehicle Type | Max Units | Service Value | Cost ($000s) | MPG |
|--------------|----------:|--------------:|-------------:|----:|
| Small        | 30        | 150           | 20           | 35  |
| Medium       | 25        | 200           | 25           | 25  |
| Large        | 20        | 250           | 30           | 20  |
| SUV          | 20        | 300           | 35           | 15  |

---

## Requirements

- Total budget available: **$1,500 ($000s)**, or **$1.5 million**
- The fleet must have an average MPG of at least **25**

---

## Model Objective

Maximize:

150(Small) + 200(Medium) + 250(Large) + 300(SUV)

---

## Constraints

**Budget constraint:**

20(Small) + 25(Medium) + 30(Large) + 35(SUV) <= 1500

**Average MPG constraint:**<br>
Note: convert the average MPG into a linear expression<br><br>
10(Small) - 5(Large) - 10(SUV) >= 0

**Bounds:**

0 <= Small <= 30  
0 <= Medium <= 25  
0 <= Large <= 20  
0 <= SUV <= 20

---

## Task

Build and solve this model in Python. Then use the solver output to answer the following questions. You can use your vibe-coded LP application from the last module to verify the answers, or use it for the exam if you trust it.

---

## Starter Code

In [2]:
import numpy as np
from ortools.linear_solver import pywraplp as glp    # import Glop package

try:
    from bana4095 import lptools as lpt
except ModuleNotFoundError:
    class lpt:
        @staticmethod
        def print_model(model):
            print(model.ExportModelAsLpFormat(False))

In [3]:
#Create LP model object
mymodel = glp.Solver.CreateSolver('GLOP')
inf = mymodel.infinity()

In [4]:
mymodel.Objective().SetMaximization()    # set the direction of optimization

variables = {
    'Small': (0, 30, 150),
    'Medium': (0, 25, 200),
    'Large': (0, 20, 250),
    'SUV': (0, 20, 300)
}

constraints = {
    'Budget': ([20, 25, 30, 35], '<=', 1500),
    'MinAvgMPG': ([10, 0, -5, -10], '>=', 0)
}

In [5]:
# Create the decision variables and objective function
for vname in variables:                               # for each variable in the variables dictionary
    (lb, ub, coeff) = variables[vname]                # retrieve the variable's parameters from the variables dictionary
    var = mymodel.NumVar(lb, ub, vname)               # create the decision variable object
    mymodel.Objective().SetCoefficient(var, coeff)    # add the variable and its coefficient to the objective function

In [6]:
# Create the constraints
var_lst = mymodel.variables()    # retrieve the list of decision variable objects
for cname in constraints:        # for each constraint in the constraints dictionary
    (coeff_lst, relation, rhs) = constraints[cname]                      # retrieve the constraint's parameters from the constraints dictionary
    lhs = sum([coeff_lst[i]*var_lst[i] for i in range(len(var_lst))])    # create the linear formula for the constraint's left-hand-side
    if relation == '<=': mymodel.Add(lhs <= rhs, cname)                  # add the constraint using the appropriate relationship
    elif relation == '==': mymodel.Add(lhs == rhs, cname)
    elif relation == '>=': mymodel.Add(lhs >= rhs, cname)
    else: print(f'Constraint {cname} has invalid relation operator {relation}')

In [7]:
lpt.print_model(mymodel)

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 2
\   Variables        : 4
\     Binary         : 0
\     Integer        : 0
\     Continuous     : 4
Maximize
 Obj: +150 Small +200 Medium +250 Large +300 SUV 
Subject to
 Budget: +20 Small +25 Medium +30 Large +35 SUV  <= 1500
 MinAvgMPG: +10 Small -5 Large -10 SUV  >= 0
Bounds
 0 <= Small <= 30
 0 <= Medium <= 25
 0 <= Large <= 20
 0 <= SUV <= 20
End



In [8]:
#solve model and display results
status = mymodel.Solve()
print(f'Solution Status = {status}')
print(f'Optimal Value = {mymodel.Objective().Value():,.2f}')
for v in mymodel.variables():
    print(f'{v.name():s} = {v.solution_value():.2f}')

Solution Status = 0
Optimal Value = 12,250.00
Small = 25.00
Medium = 0.00
Large = 10.00
SUV = 20.00


In [9]:
# display variable information
print('Variable    LB   Value    UB   Reduced Cost')
for v in mymodel.variables():
    print(f'{v.name():s}  {v.lb():.1f}  {v.solution_value():.1f}  {v.ub():.1f}  {v.reduced_cost():.2f}')

Variable    LB   Value    UB   Reduced Cost
Small  0.0  25.0  30.0  0.00
Medium  0.0  0.0  25.0  -3.12
Large  0.0  10.0  20.0  0.00
SUV  0.0  20.0  20.0  3.12


In [10]:
#display constraint information
print('Constraint    LB    Value  UB     Dual')
for i in range(len(mymodel.constraints())):
    c = mymodel.constraints()[i]
    lhs = mymodel.ComputeConstraintActivities()[i]
    print(f'{c.name():s}  {c.lb():.1f}  {lhs:.1f}  {c.ub():.1f}  {c.dual_value():.2f}')

Constraint    LB    Value  UB     Dual
Budget  -inf  1500.0  1500.0  8.12
MinAvgMPG  0.0  0.0  inf  -1.25


In [11]:
# Helper analysis for Questions 6-9
base_solution = {v.name(): v.solution_value() for v in mymodel.variables()}
service = {'Small': 150, 'Medium': 200, 'Large': 250, 'SUV': 300}
cost = {'Small': 20, 'Medium': 25, 'Large': 30, 'SUV': 35}
mpg = {'Small': 35, 'Medium': 25, 'Large': 20, 'SUV': 15}

base_units = sum(base_solution.values())
base_budget_used = sum(cost[k] * base_solution[k] for k in base_solution)
base_avg_cost = base_budget_used / base_units if base_units else 0

print(f"Base total units = {base_units:.0f}")
print(f"Base budget used ($000s) = {base_budget_used:.2f}")
print(f"Base average cost per vehicle ($000s) = {base_avg_cost:.4f}")


def solve_fleet(total_budget, min_avg_mpg):
    m = glp.Solver.CreateSolver('GLOP')
    x = {
        'Small': m.NumVar(0, 30, 'Small'),
        'Medium': m.NumVar(0, 25, 'Medium'),
        'Large': m.NumVar(0, 20, 'Large'),
        'SUV': m.NumVar(0, 20, 'SUV')
    }

    # Objective: maximize service value
    m.Maximize(sum(service[k] * x[k] for k in x))

    # Budget and average-MPG constraints
    m.Add(sum(cost[k] * x[k] for k in x) <= total_budget, 'Budget')
    m.Add(sum((mpg[k] - min_avg_mpg) * x[k] for k in x) >= 0, 'MinAvgMPG')

    status = m.Solve()
    sol = {k: x[k].solution_value() for k in x}
    obj = m.Objective().Value()
    used = sum(cost[k] * sol[k] for k in sol)
    return status, obj, sol, used

status_15, obj_15, sol_15, used_15 = solve_fleet(1500, 25)
status_30, obj_30, sol_30, used_30 = solve_fleet(3000, 25)
status_30_30, obj_30_30, sol_30_30, used_30_30 = solve_fleet(3000, 30)

print(f"\nQ8 baseline (Budget=1500, MPG>=25): {obj_15:.2f}")
print(f"Q8 new (Budget=3000, MPG>=25): {obj_30:.2f}")
print(f"Q9 new (Budget=3000, MPG>=30): {obj_30_30:.2f}")

print("\nMix at Budget=3000, MPG>=30")
for k, v in sol_30_30.items():
    print(f"{k}: {v:.2f}")

Base total units = 55
Base budget used ($000s) = 1500.00
Base average cost per vehicle ($000s) = 27.2727

Q8 baseline (Budget=1500, MPG>=25): 12250.00
Q8 new (Budget=3000, MPG>=25): 20500.00
Q9 new (Budget=3000, MPG>=30): 10125.00

Mix at Budget=3000, MPG>=30
Small: 30.00
Medium: 25.00
Large: 2.50
SUV: 0.00


In [12]:
# Integer (MILP) version for Question 10 scenario: budget=3000, min average MPG=30
mip = glp.Solver.CreateSolver('CBC')

x_int = {
    'Small': mip.IntVar(0, 30, 'Small'),
    'Medium': mip.IntVar(0, 25, 'Medium'),
    'Large': mip.IntVar(0, 20, 'Large'),
    'SUV': mip.IntVar(0, 20, 'SUV')
}

mip.Maximize(sum(service[k] * x_int[k] for k in x_int))
mip.Add(sum(cost[k] * x_int[k] for k in x_int) <= 3000, 'Budget_3M')
mip.Add(sum((mpg[k] - 30) * x_int[k] for k in x_int) >= 0, 'MinAvgMPG_30')

mip_status = mip.Solve()
mip_solution = {k: x_int[k].solution_value() for k in x_int}
mip_obj = mip.Objective().Value()
mip_budget_used = sum(cost[k] * mip_solution[k] for k in mip_solution)

print(f'MIP Status = {mip_status}')
print(f'MIP Optimal Service Value = {mip_obj:.2f}')
print(f'MIP Budget Used ($000s) = {mip_budget_used:.2f}')
for k, v in mip_solution.items():
    print(f'{k}: {v:.0f}')

MIP Status = 0
MIP Optimal Service Value = 10050.00
MIP Budget Used ($000s) = 1290.00
Small: 30
Medium: 24
Large: 3
SUV: 0


---
## Question 1

What is the optimal (maximum) total annual service value?

**Answer:** **12,250**

## Question 2

How many of each **COMPACT** vehicles should be purchased in the optimal solution?

**Answer:** **25 compact vehicles**

## Question 3

How many of each **SUDAN** vehicles should be purchased in the optimal solution?

**Answer:** **0 sedan vehicles**

## Question 4

How many of each **PICKUP** vehicles should be purchased in the optimal solution?

**Answer:** **10 pickup trucks**

## Question 5

How many of each **SUV** vehicles should be purchased in the optimal solution?

**Answer:** **20 SUVs**

## Question 6

What is the average cost per vehicle in the optimal solution?

**Answer:**
- Average cost = 1500 / 55 = **27.2727 ($000s)**
- Equivalent to about **$27,273 per vehicle**

## Question 7

How much of the total budget is used in the optimal solution?

**Answer:**
- Budget used = **1,500 ($000s)** = **$1.5 million**
- This is **100%** of the available budget

## Question 8

How does the optimal value change as the total budget increases from $1.5M to $3.0M?

**Answer:**
- At $1.5M: optimal service value = **12,250**
- At $3.0M: optimal service value = **20,500**
- Change = **+8,250** (about **+67.35%**)

## Question 9

With the new $3M budget, if the minimum average fuel economy requirement increases by only 10% from 25 MPG to 30 MPG, what is the new optimal total service value?

**Answer:** **10,125**

## Question 10

Under the new $3M budget and 30 MPG requirement, is the optimal solution for the vehicle mix a realistic plan for the city? Explain why or why not.

**Answer:** A realistic plan should use integer (whole-number) vehicles, so I re-solved this sce    nario as a MILP. The whole-vehicle optimum is:
- Small = **30**
- Medium = **24**
- Large (Pickup) = **3**
- SUV = **0**
- Total service value = **10,050**

This is realistic from a counting standpoint (no fractional vehicles), but it still may not be fully operationally realistic because it leaves out SUVs and uses only a small number of pickups, which could be insufficient for heavy-duty or specialized city tasks.